
## Librarie's Installation


In [1]:
!pip -q install -U "transformers==5.1.0" "accelerate>=1.0.0" datasets evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00


## Connection in google drive

In [2]:
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/sentiment_analysis_finetuning"
except Exception:
    SAVE_DIR = "."

os.makedirs(SAVE_DIR, exist_ok=True)
print("Results:", SAVE_DIR)

Mounted at /content/drive
Results: /content/drive/MyDrive/sentiment_analysis_finetuning


## Load IMDB dataset

In [3]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")
print(imdb)
print("\nTrain size:", len(imdb["train"]), " ,  Test size:", len(imdb["test"]))
print("Example:", imdb["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Train size: 25000  ,  Test size: 25000
Example: {'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the

## Dataset split

In [4]:
# Χωρίζουμε το train σε train (90%) + validation (10%)
split = imdb["train"].train_test_split(test_size=0.1, seed=42)
imdb["train"] = split["train"]
imdb["validation"] = split["test"]

print("  Train size:", len(imdb["train"]))
print("  Validation size:", len(imdb["validation"]))
print("  Test size:", len(imdb["test"]))

  Train size: 22500
  Validation size: 2500
  Test size: 25000


## Tokenization

In [5]:
from transformers import AutoTokenizer

model_name_bert = "google-bert/bert-base-uncased"
tokenizer_bert = AutoTokenizer.from_pretrained(model_name_bert)

def tokenize_function(examples, tokenizer, max_length=512):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )

def tokenize_bert(examples):
    return tokenize_function(examples, tokenizer_bert)

# Tokenize train και test για BERT
imdb_bert = imdb.map(
    tokenize_bert,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing (BERT)",
)
imdb_bert.set_format("torch")
print("Train:", len(imdb_bert["train"]), "| Validation:", len(imdb_bert["validation"]), "| Test:", len(imdb_bert["test"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing (BERT):   0%|          | 0/22500 [00:00<?, ? examples/s]

Tokenizing (BERT):   0%|          | 0/25000 [00:00<?, ? examples/s]

Tokenizing (BERT):   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing (BERT):   0%|          | 0/2500 [00:00<?, ? examples/s]

Train: 22500 | Validation: 2500 | Test: 25000


## Finetuning training

In [6]:
import os
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
from transformers import DataCollatorWithPadding
data_collator_bert = DataCollatorWithPadding(tokenizer=tokenizer_bert)


metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

model_bert = AutoModelForSequenceClassification.from_pretrained(
    model_name_bert,
    num_labels=2,
)

training_args = TrainingArguments(
    output_dir=os.path.join(SAVE_DIR, "results_bert"),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    report_to="none",
)

trainer_bert = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=imdb_bert["train"],
    eval_dataset=imdb_bert["validation"],
    data_collator=data_collator_bert,
    compute_metrics=compute_metrics,
)



trainer_bert.train()


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.296412,0.310950,0.917600
2,0.161280,0.293735,0.934400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5626, training_loss=0.25893811064176697, metrics={'train_runtime': 1333.7659, 'train_samples_per_second': 33.739, 'train_steps_per_second': 4.218, 'total_flos': 1.18399974912e+16, 'train_loss': 0.25893811064176697, 'epoch': 2.0})

## Evaluation on Test accuracy

In [7]:
# test accuracy
eval_bert = trainer_bert.evaluate(eval_dataset=imdb_bert["test"])
print("BERT base - Test accuracy:", eval_bert["eval_accuracy"])

# save metrics
import json
with open(os.path.join(SAVE_DIR, "results_bert_metrics.json"), "w") as f:
    json.dump({k: float(v) if hasattr(v, "item") else v for k, v in eval_bert.items()}, f, indent=2)
print("Saved", os.path.join(SAVE_DIR, "results_bert_metrics.json"))

BERT base - Test accuracy: 0.93608
Saved /content/drive/MyDrive/sentiment_analysis_finetuning/results_bert_metrics.json


## Evaluation on F1 metric

In [8]:
import numpy as np
import evaluate

f1_metric = evaluate.load("f1")

def compute_f1_only(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]}

trainer_bert.compute_metrics = compute_f1_only

eval_f1 = trainer_bert.evaluate(imdb_bert["test"])
print("BERT base test in F1:", eval_f1["eval_f1"])


BERT base test in F1: 0.936627538071066
